# Post 1: comparing retrievers

BM25, three embedding models (bge-m3, EmbeddingGemma 2, Qwen3-Embedding-0.6B) and
BM25 fused with each model by RRF, on the 54 shortlist questions in two wordings:
as written, and paraphrased blind (see `PLAN.md`).

This notebook only reads saved results, so it runs in seconds. After changing a
retriever or the eval set, regenerate them:

```
uv run run-eval --save
uv run run-eval --paraphrase --save
```

Hit@k counts a question as found if any source post is in the top k. With 54
questions, one question is about 0.02 of an "all" score, and one of the 8
current_state questions is 0.125.

In [1]:
import pandas as pd
from IPython.display import Markdown, display

from retrieval_bench.analysis import (agreement, by_retriever, paraphrase_flips,
                                      random_baseline, term_contributions)
from retrieval_bench.evaluate import summarize
from retrieval_bench.models import RESULTS_DIR, load_questions, load_results
from retrieval_bench.retrieval import load_posts
from retrieval_bench.retrievers import EMBEDDING_MODELS, BM25Retriever

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 200)

questions = {q.id: q for q in load_questions()}
posts = load_posts()
WORDINGS = ("original", "paraphrase")
results = {w: load_results(RESULTS_DIR / f"{w}.json") for w in WORDINGS}
table = {w: by_retriever(results[w]) for w in WORDINGS}
RETRIEVERS = list(table["original"])
MODELS = list(EMBEDDING_MODELS)

def text(qid, wording):
    q = questions[qid]
    return q.paraphrase if wording == "paraphrase" else q.question

print(f"{len(questions)} questions, {len(posts)} posts, retrievers: {RETRIEVERS}")

54 questions, 44 posts, retrievers: ['bm25', 'bge-m3', 'gemma2', 'qwen3-0.6b', 'rrf(bm25+bge-m3)', 'rrf(bm25+gemma2)', 'rrf(bm25+qwen3-0.6b)']


/Users/bryan/Projects/retrieval-bench-blog-qa-agent/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Overall scores

All 54 questions, both wordings, with the expected score of a random ranking
(computed from each question's number of source posts).

In [2]:
summary = pd.DataFrame([
    {"wording": w, "retriever": s.retriever, "group": s.group, "n": s.n,
     **{f"Hit@{k}": v for k, v in s.hit_at.items()}, "MRR": s.mrr}
    for w in WORDINGS for s in summarize(results[w])])

overall = (summary[summary.group == "all"]
           .pivot(index="retriever", columns="wording", values=["Hit@1", "Hit@3", "MRR"])
           .reindex(RETRIEVERS))
base = random_baseline(list(questions.values()), len(posts))
overall.loc["random"] = [base["mrr"] if metric == "MRR" else base["hit_at"][int(metric[4:])]
                         for metric, _ in overall.columns]
overall.round(2)

Hit@1               Hit@3                 MRR           
wording              original paraphrase original paraphrase original paraphrase
retriever                                                                       
bm25                     0.78       0.72     0.91       0.89     0.85       0.81
bge-m3                   0.78       0.74     0.91       0.91     0.85       0.83
gemma2                   0.85       0.83     0.96       0.96     0.92       0.90
qwen3-0.6b               0.80       0.83     0.93       0.89     0.86       0.88
rrf(bm25+bge-m3)         0.85       0.81     0.98       0.94     0.91       0.88
rrf(bm25+gemma2)         0.87       0.81     1.00       0.94     0.92       0.88
rrf(bm25+qwen3-0.6b)     0.83       0.80     0.98       0.94     0.90       0.87
random                   0.03       0.03     0.09       0.09     0.12       0.12

### By question type

Hit@1 per type. single_fact (35) dominates the "all" row; current_state (8)
is where every retriever struggles.

In [3]:
by_type = (summary[summary.group != "all"]
           .pivot(index="retriever", columns=["wording", "group"], values="Hit@1")
           .reindex(RETRIEVERS))
by_type.round(2)

wording                 original                                   paraphrase                                 
group                single_fact current_state multi_post opinion single_fact current_state multi_post opinion
retriever                                                                                                     
bm25                        0.89          0.12        0.8     1.0        0.83          0.12        1.0    0.67
bge-m3                      0.86          0.38        0.6     1.0        0.89          0.12        0.4    1.00
gemma2                      0.94          0.38        0.8     1.0        0.94          0.25        0.8    1.00
qwen3-0.6b                  0.89          0.25        0.8     1.0        0.94          0.25        0.8    1.00
rrf(bm25+bge-m3)            0.94          0.38        0.8     1.0        0.94          0.25        0.8    0.83
rrf(bm25+gemma2)            0.97          0.25        1.0     1.0        0.94          0.25        0.8    0.83
rrf(bm25+qwen3-0.6b)        0.94          0.12        1.0     1.0        0.94          0.12        0.8    0.83

## 2. Where BM25 and each embedding model disagree

At Hit@1: does each retriever put a source post first? The "all" scores can tie
while the two retrievers get different questions right, so this counts each
cell of the 2×2 table.

In [4]:
rows = []
for w in WORDINGS:
    for m in MODELS:
        cells = agreement(results[w], "bm25", m, k=1)
        rows.append({"wording": w, "model": m,
                     "both hit": len(cells[True, True]),
                     "bm25 only": len(cells[True, False]),
                     "model only": len(cells[False, True]),
                     "neither": len(cells[False, False]),
                     "bm25 only ids": " ".join(cells[True, False]),
                     "model only ids": " ".join(cells[False, True]),
                     "neither ids": " ".join(cells[False, False])})
pd.DataFrame(rows).set_index(["wording", "model"])

both hit  bm25 only  model only  neither                  bm25 only ids                                model only ids                                   neither ids
wording    model                                                                                                                                                                          
original   bge-m3            39          3           3        9                 sf32 sf67 mp01                                sf66 cs12 cs18  sf33 sf59 sf75 cs01 cs04 cs13 cs17 cs21 mp08
           gemma2            42          0           4        8                                                          sf33 sf59 cs12 cs18       sf66 sf75 cs01 cs04 cs13 cs17 cs21 mp08
           qwen3-0.6b        40          2           3        9                      sf01 sf31                                sf33 sf59 cs01  sf66 sf75 cs04 cs12 cs13 cs17 cs18 cs21 mp08
paraphrase bge-m3            33          6           7        8  sf32 sf67 cs22 mp01 mp03 mp08            sf05 sf59 sf60 sf66 cs18 op14 op18       sf33 sf75 cs01 cs04 cs12 cs13 cs17 cs21
           gemma2            36          3           9        6                 sf01 cs22 mp08  sf05 sf33 sf59 sf60 sf66 cs12 cs18 op14 op18                 sf75 cs01 cs04 cs13 cs17 cs21
           qwen3-0.6b        37          2           8        7                      sf01 mp08       sf05 sf33 sf59 sf60 sf66 cs01 op14 op18            sf75 cs04 cs12 cs13 cs17 cs18 cs21

The same comparison as a 2×2 table, for one pair (change `WORDING` and `MODEL`):

In [5]:
WORDING, MODEL = "original", "gemma2"
cells = agreement(results[WORDING], "bm25", MODEL, k=1)
pd.DataFrame([[len(cells[True, True]), len(cells[True, False])],
              [len(cells[False, True]), len(cells[False, False])]],
             index=[f"bm25 hit", f"bm25 miss"], columns=[f"{MODEL} hit", f"{MODEL} miss"])

,gemma2 hit,gemma2 miss
bm25 hit,42,0
bm25 miss,4,8


## 3. Reading the disagreements

Each question one retriever gets at Hit@1 and the other misses: its top 3 from
both. `*` marks a source post; `first` is the rank of the first source post.

In [6]:
def top(result, n=3):
    sources = questions[result.question_id].source_posts
    return ", ".join(("*" if s in sources else "") + s for s in result.ranking[:n])

def disagreements(wording, model, k=1):
    cells = agreement(results[wording], "bm25", model, k)
    rows = []
    for winner, qids in (("bm25", cells[True, False]), (model, cells[False, True])):
        for qid in qids:
            b, e = table[wording]["bm25"][qid], table[wording][model][qid]
            rows.append({"id": qid, "type": questions[qid].type, "wins": winner,
                         "question": text(qid, wording),
                         "bm25 first": b.first_source_rank, "bm25 top 3": top(b),
                         f"{model} first": e.first_source_rank, f"{model} top 3": top(e)})
    return pd.DataFrame(rows)

for w in WORDINGS:
    for m in MODELS:
        display(Markdown(f"**{w}: bm25 vs {m}**"))
        display(disagreements(w, m))

**original: bm25 vs bge-m3**

,id,type,wins,question,bm25 first,bm25 top 3,bge-m3 first,bge-m3 top 3
0,sf32,single_fact,bm25,What open question did Bryan end his token-pricing post with?,1,"*one-reason-input-tokens-are-cheaper-than-output-tokens, shift-from-ml-to-ai-engineeri...",2,"building-a-simple-sentiment-classifier, *one-reason-input-tokens-are-cheaper-than-outp..."
1,sf67,single_fact,bm25,How did he look after his mental health while stuck at home in 2020?,1,"*making-the-most-of-my-solitude, 2020-in-review, stephen-king-lesson-failure",2,"2020-in-review, *making-the-most-of-my-solitude, 2022-in-review"
2,mp01,multi_post,bm25,How did Bryan's homelab setup change over 2026?,1,"*building-a-better-nas-for-my-homelab, *my-homelab-setup, home-vpn-dynamic-dns",3,"2020-in-review, 2022-in-review, *my-homelab-setup"
3,sf66,single_fact,bge-m3,What kept Bryan from posting more often in his blog?,4,"2020-in-review, stephen-king-lesson-failure, thinking-about-the-value-of-writing",1,"*just-write-more, rethinking-my-relationship-with-instagram, making-the-most-of-my-sol..."
4,cs12,current_state,bge-m3,Does Bryan's website have a project's page still?,5,"rethinking-my-relationship-with-instagram, creating-sveltekit-sitemap, redesign-5",1,"*redesign-4, hello-world, redesign-3"
5,cs18,current_state,bge-m3,What operating system does Bryan use in his VPS?,2,"self-hosting-umami-analytics, *black-october-made-me-switch-from-ubuntu-to-debian, ins...",1,"*black-october-made-me-switch-from-ubuntu-to-debian, building-a-better-nas-for-my-home..."


**original: bm25 vs gemma2**

,id,type,wins,question,bm25 first,bm25 top 3,gemma2 first,gemma2 top 3
0,sf33,single_fact,gemma2,"Which NAS cases did Bryan consider besides the Sagittarius, and why did he reject those?",2,"building-a-better-nas-for-my-homelab, *sagittarius-8-bay-nas-first-impressions, stephe...",1,"*sagittarius-8-bay-nas-first-impressions, building-a-better-nas-for-my-homelab, intro-..."
1,sf59,single_fact,gemma2,How does Bryan trim his backlog at the start of each month?,2,"2020-in-review, *getting-things-done-with-things-3, creating-sveltekit-sitemap",1,"*getting-things-done-with-things-3, 2024-in-review, 2023-in-review"
2,cs12,current_state,gemma2,Does Bryan's website have a project's page still?,5,"rethinking-my-relationship-with-instagram, creating-sveltekit-sitemap, redesign-5",1,"*redesign-4, redesign-3, redesign-2"
3,cs18,current_state,gemma2,What operating system does Bryan use in his VPS?,2,"self-hosting-umami-analytics, *black-october-made-me-switch-from-ubuntu-to-debian, ins...",1,"*black-october-made-me-switch-from-ubuntu-to-debian, self-hosting-umami-analytics, my-..."


**original: bm25 vs qwen3-0.6b**

,id,type,wins,question,bm25 first,bm25 top 3,qwen3-0.6b first,qwen3-0.6b top 3
0,sf01,single_fact,bm25,In one of Bryan's blog posts he timed how long it takes to process input tokens and ou...,1,"*one-reason-input-tokens-are-cheaper-than-output-tokens, straight-line-predicts-how-fa...",2,"straight-line-predicts-how-fast-my-laptop-generates-llm-tokens, *one-reason-input-toke..."
1,sf31,single_fact,bm25,"When Bryan fired 20 rapid calls at his throttled endpoint, why did one late call get t...",1,"*implementing-rate-limiter-fastapi-redis, home-vpn-dynamic-dns, creating-simple-rest-a...",3,"straight-line-predicts-how-fast-my-laptop-generates-llm-tokens, fixing-claude-code-api..."
2,sf33,single_fact,qwen3-0.6b,"Which NAS cases did Bryan consider besides the Sagittarius, and why did he reject those?",2,"building-a-better-nas-for-my-homelab, *sagittarius-8-bay-nas-first-impressions, stephe...",1,"*sagittarius-8-bay-nas-first-impressions, building-a-better-nas-for-my-homelab, 2020-i..."
3,sf59,single_fact,qwen3-0.6b,How does Bryan trim his backlog at the start of each month?,2,"2020-in-review, *getting-things-done-with-things-3, creating-sveltekit-sitemap",1,"*getting-things-done-with-things-3, 2022-in-review, making-the-most-of-my-solitude"
4,cs01,current_state,qwen3-0.6b,Does Bryan's website still use Tailwind CSS?,3,"simplifying-my-website-css-using-tailwind, configuring-tailwind-css-v4, *redesign-5",1,"*redesign-5, configuring-tailwind-css-v4, simplifying-my-website-css-using-tailwind"


... (2 outputs left out)

**paraphrase: bm25 vs gemma2**

,id,type,wins,question,bm25 first,bm25 top 3,gemma2 first,gemma2 top 3
0,sf01,single_fact,bm25,Bryan timed how fast LLMs on his laptop read the prompt compared with how fast they wr...,1,"*one-reason-input-tokens-are-cheaper-than-output-tokens, autogen-weather-chatbot, stra...",2,"straight-line-predicts-how-fast-my-laptop-generates-llm-tokens, *one-reason-input-toke..."
1,cs22,current_state,bm25,Why does the smallest Qwen model take about 9 ms for each token it generates?,1,"*straight-line-predicts-how-fast-my-laptop-generates-llm-tokens, one-reason-input-toke...",2,"one-reason-input-tokens-are-cheaper-than-output-tokens, *straight-line-predicts-how-fa..."
2,mp08,multi_post,bm25,Which places in the Dolomites has he posted photos from?,1,"*2024-in-review, 2022-in-review, rethinking-my-relationship-with-instagram",3,"2022-in-review, rethinking-my-relationship-with-instagram, *2024-in-review"
3,sf05,single_fact,gemma2,What method did Bryan use to cap how many requests people can make?,3,"my-first-visit-to-chicago, custom-celery-mongodb-backend, *implementing-rate-limiter-f...",1,"*implementing-rate-limiter-fastapi-redis, creating-simple-rest-api-elixir-phoenix, 202..."
4,sf33,single_fact,gemma2,"Besides the Sagittarius, what other NAS cases did Bryan look at, and why did he pass o...",2,"building-a-better-nas-for-my-homelab, *sagittarius-8-bay-nas-first-impressions, making...",1,"*sagittarius-8-bay-nas-first-impressions, building-a-better-nas-for-my-homelab, intro-..."
5,sf59,single_fact,gemma2,What does Bryan do at the start of each month to cut down his to-do pile?,4,"sagittarius-8-bay-nas-first-impressions, 2020-in-review, self-hosting-umami-analytics",1,"*getting-things-done-with-things-3, 2023-in-review, 2024-in-review"
6,sf60,single_fact,gemma2,What did Bryan build the first version of his site with?,3,"self-hosting-umami-analytics, making-the-most-of-my-solitude, *2020-in-review",1,"*hello-world, *redesign-2, redesign-4"
7,sf66,single_fact,gemma2,What held Bryan back from putting out blog posts more often?,4,"my-first-steps-in-photography, redesign-4, 2020-in-review",1,"*just-write-more, rethinking-my-relationship-with-instagram, thinking-about-the-value-..."
8,cs12,current_state,gemma2,Is there still a projects page on Bryan's site?,11,"redesign-5, creating-sveltekit-sitemap, rethinking-my-relationship-with-instagram",1,"*redesign-4, redesign-3, redesign-2"
9,cs18,current_state,gemma2,What OS is Bryan running on his VPS?,2,"self-hosting-umami-analytics, *black-october-made-me-switch-from-ubuntu-to-debian, ste...",1,"*black-october-made-me-switch-from-ubuntu-to-debian, my-homelab-setup, self-hosting-um..."


**paraphrase: bm25 vs qwen3-0.6b**

,id,type,wins,question,bm25 first,bm25 top 3,qwen3-0.6b first,qwen3-0.6b top 3
0,sf01,single_fact,bm25,Bryan timed how fast LLMs on his laptop read the prompt compared with how fast they wr...,1,"*one-reason-input-tokens-are-cheaper-than-output-tokens, autogen-weather-chatbot, stra...",2,"straight-line-predicts-how-fast-my-laptop-generates-llm-tokens, *one-reason-input-toke..."
1,mp08,multi_post,bm25,Which places in the Dolomites has he posted photos from?,1,"*2024-in-review, 2022-in-review, rethinking-my-relationship-with-instagram",9,"2022-in-review, hello-world, rethinking-my-relationship-with-instagram"
2,sf05,single_fact,qwen3-0.6b,What method did Bryan use to cap how many requests people can make?,3,"my-first-visit-to-chicago, custom-celery-mongodb-backend, *implementing-rate-limiter-f...",1,"*implementing-rate-limiter-fastapi-redis, straight-line-predicts-how-fast-my-laptop-ge..."
3,sf33,single_fact,qwen3-0.6b,"Besides the Sagittarius, what other NAS cases did Bryan look at, and why did he pass o...",2,"building-a-better-nas-for-my-homelab, *sagittarius-8-bay-nas-first-impressions, making...",1,"*sagittarius-8-bay-nas-first-impressions, building-a-better-nas-for-my-homelab, 2020-i..."
4,sf59,single_fact,qwen3-0.6b,What does Bryan do at the start of each month to cut down his to-do pile?,4,"sagittarius-8-bay-nas-first-impressions, 2020-in-review, self-hosting-umami-analytics",1,"*getting-things-done-with-things-3, making-the-most-of-my-solitude, 2024-in-review"
5,sf60,single_fact,qwen3-0.6b,What did Bryan build the first version of his site with?,3,"self-hosting-umami-analytics, making-the-most-of-my-solitude, *2020-in-review",1,"*hello-world, *redesign-2, redesign-4"
6,sf66,single_fact,qwen3-0.6b,What held Bryan back from putting out blog posts more often?,4,"my-first-steps-in-photography, redesign-4, 2020-in-review",1,"*just-write-more, making-the-most-of-my-solitude, rethinking-my-relationship-with-inst..."
7,cs01,current_state,qwen3-0.6b,Is Bryan's site still styled with Tailwind CSS?,3,"simplifying-my-website-css-using-tailwind, configuring-tailwind-css-v4, *redesign-5",1,"*redesign-5, configuring-tailwind-css-v4, simplifying-my-website-css-using-tailwind"
8,op14,opinion,qwen3-0.6b,How does Bryan think writing for school compares to writing for his blog?,2,"2020-in-review, *thinking-about-the-value-of-writing, stephen-king-lesson-failure",1,"*thinking-about-the-value-of-writing, just-write-more, making-the-most-of-my-solitude"
9,op18,opinion,qwen3-0.6b,What's Bryan's explanation for why the tokens a model generates cost more than the one...,2,"straight-line-predicts-how-fast-my-laptop-generates-llm-tokens, *one-reason-input-toke...",1,"*one-reason-input-tokens-are-cheaper-than-output-tokens, straight-line-predicts-how-fa..."


## 4. What the paraphrases change

Per retriever, the questions that go from a hit to a miss at Hit@1 when
reworded (lost), and the reverse (gained).

In [7]:
rows = []
for r in RETRIEVERS:
    lost, gained = paraphrase_flips(results["original"], results["paraphrase"], r, k=1)
    rows.append({"retriever": r, "lost": len(lost), "gained": len(gained),
                 "lost ids": " ".join(lost), "gained ids": " ".join(gained)})
pd.DataFrame(rows).set_index("retriever")

,lost,gained,lost ids,gained ids
retriever,,,,
bm25,4,1,sf05 sf60 op14 op18,mp08
bge-m3,3,1,cs12 cs22 mp03,sf59
gemma2,2,1,sf01 cs22,sf66
qwen3-0.6b,0,2,,sf31 sf66
rrf(bm25+bge-m3),2,0,cs12 op18,
rrf(bm25+gemma2),4,1,sf75 cs12 mp08 op18,cs18
rrf(bm25+qwen3-0.6b),2,0,mp08 op18,


## 5. current_state questions

The rank of the first current post for each retriever. `S` marks a superseded
post (one with an outdated answer) ranked first. These misses are about time,
not wording, which is the topic of a later post.

In [8]:
cs = [qid for qid, q in questions.items() if q.type == "current_state"]

def cell(result):
    stale = result.ranking[0] in questions[result.question_id].superseded_posts
    return f"{result.first_source_rank}{' S' if stale else ''}"

for w in WORDINGS:
    display(Markdown(f"**{w}**"))
    display(pd.DataFrame({r: {qid: cell(table[w][r][qid]) for qid in cs} for r in RETRIEVERS}))

**original**

,bm25,bge-m3,gemma2,qwen3-0.6b,rrf(bm25+bge-m3),rrf(bm25+gemma2),rrf(bm25+qwen3-0.6b)
cs01,3 S,3 S,2 S,1,3 S,3 S,2 S
cs04,6 S,4 S,4 S,17,3 S,3 S,9 S
cs12,5,1,1,3,1,1,2 S
cs13,3 S,5 S,2,6,3 S,3,3 S
cs17,4,5 S,5 S,2 S,3 S,3,2 S
cs18,2 S,1,1,3,1,2 S,2 S
cs21,3,2 S,2 S,3 S,2 S,3 S,3 S
cs22,1,1,1,1,1,1,1


**paraphrase**

,bm25,bge-m3,gemma2,qwen3-0.6b,rrf(bm25+bge-m3),rrf(bm25+gemma2),rrf(bm25+qwen3-0.6b)
cs01,3 S,3 S,2 S,1,3 S,3 S,2 S
cs04,5 S,11 S,6 S,22 S,5 S,4 S,12 S
cs12,11,2,1,4 S,7,5,7
cs13,3 S,4 S,2,4,3 S,3,3 S
cs17,2 S,12,4,2 S,3 S,3 S,2 S
cs18,2 S,1,1,3,1,1,2 S
cs21,4 S,2 S,2 S,4 S,2 S,3 S,3 S
cs22,1,2 S,2 S,1,1,1,1


## 6. Why BM25 missed: the score, term by term

A BM25 score is a sum over the query's terms (after stopwords and stemming), so
each miss can be explained by comparing what each term adds to the post BM25
put first and to the highest-ranked source post. `posts with term` is the
document frequency: a term in few posts has a high IDF and counts for more.
Rows are sorted by how much each term favors the wrong post.

In [9]:
bm25 = BM25Retriever(posts)

def explain(qid, wording="original"):
    r = table[wording]["bm25"][qid]
    sources = questions[qid].source_posts
    wrong = r.ranking[0]
    right = next(s for s in r.ranking if s in sources)
    df = pd.DataFrame(term_contributions(bm25, text(qid, wording), [wrong, right]))
    df = df.rename(columns={"posts_with_term": "posts with term",
                            wrong: f"top: {wrong}", right: f"source (rank {r.first_source_rank}): {right}"})
    df["favors top by"] = df.iloc[:, 3] - df.iloc[:, 4]
    df = df.sort_values("favors top by", ascending=False).set_index("term")
    df[["count", "posts with term"]] = df[["count", "posts with term"]].astype("Int64")  # blank allowed
    df.loc["TOTAL"] = df.sum(numeric_only=True)
    df.loc["TOTAL", ["count", "posts with term"]] = None
    display(Markdown(f"**{qid}** ({wording}): {text(qid, wording)}"))
    display(df.round(2))

misses = [qid for qid, r in table["original"]["bm25"].items() if not r.hit_at[1]]
print("BM25 Hit@1 misses on the original wording:", misses)
for qid in misses:
    explain(qid)

BM25 Hit@1 misses on the original wording: ['sf33', 'sf59', 'sf66', 'sf75', 'cs01', 'cs04', 'cs12', 'cs13', 'cs17', 'cs18', 'cs21', 'mp08']


**sf33** (original): Which NAS cases did Bryan consider besides the Sagittarius, and why did he reject those?

,count,posts with term,top: building-a-better-nas-for-my-homelab,source (rank 2): sagittarius-8-bay-nas-first-impressions,favors top by
term,,,,,
did,2,6,1.68,0.00,1.68
whi,1,18,0.25,0.00,0.25
consid,1,13,0.64,0.40,0.24
nas,1,3,2.33,2.27,0.06
which,1,35,0.20,0.16,0.04
bryan,1,0,0.00,0.00,0.00
besid,1,1,0.00,0.00,0.00
he,1,4,0.00,0.00,0.00
reject,1,2,0.00,0.00,0.00


**sf59** (original): How does Bryan trim his backlog at the start of each month?

,count,posts with term,top: 2020-in-review,source (rank 2): getting-things-done-with-things-3,favors top by
term,,,,,
his,1,3,0.90,0.00,0.90
start,1,28,0.31,0.22,0.09
bryan,1,0,0.00,0.00,0.00
trim,1,2,0.00,0.00,0.00
backlog,1,0,0.00,0.00,0.00
how,1,38,0.10,0.11,-0.02
each,1,32,0.17,0.26,-0.09
month,1,12,0.98,1.18,-0.20
doe,1,12,0.00,0.62,-0.62


**sf66** (original): What kept Bryan from posting more often in his blog?

... (15 outputs left out)

**cs21** (original): What is in Bryan's current homelab?

,count,posts with term,top: home-vpn-dynamic-dns,source (rank 3): building-a-better-nas-for-my-homelab,favors top by
term,,,,,
homelab,1,4,1.47,0.64,0.84
current,1,10,1.06,0.78,0.28
what,1,30,0.30,0.17,0.13
bryan,1,0,0.00,0.00,0.00
TOTAL,<NA>,<NA>,2.83,1.58,1.25


**mp08** (original): What spots in the Dolomites has he shared photos of?

,count,posts with term,top: my-homelab-setup,source (rank 3): 2024-in-review,favors top by
term,,,,,
spot,1,1,1.35,0.00,1.35
photo,1,14,0.95,0.00,0.95
has,1,30,0.28,0.15,0.13
he,1,4,0.00,0.00,0.00
share,1,22,0.28,0.39,-0.12
what,1,30,0.00,0.22,-0.22
dolomit,1,3,0.00,1.45,-1.45
TOTAL,<NA>,<NA>,2.87,2.22,0.65
